# plot_03b. Forced response figures: the quantiles and the width

The figures for the forced response, drawn from what notebook calc_02 saved, so nothing slow runs here. Notebook calc_02 explains each method, step by step. The figures are in four notebooks, so that none grows too big to open; each opens only the saved results it uses, so they can be run in any order after notebook calc_02:

| Notebook | Sections |
|---|---|
| plot_03a | 1. Mean change; 2. Signal to noise (and a check of every model's headline numbers) |
| plot_03b | 3. Quantile response; 4. Change in width, and which tail |
| plot_03c | 5. The whole distribution; 6. Additivity; 7. Zonal means and the sea-ice edge |
| plot_03d | 8. Probability ratios; 9. The shift-only ratio |

The sections keep one numbering across the four.

Every section shows its result in the same four figures, with the four seasons across every map:

1. **What goes in.** The quantities the result is made from, then the result, one row each: one model and experiment (`EXAMPLE_MODEL`, `EXPERIMENT`).
2. **One model.** The parts and the result for every experiment of `EXAMPLE_MODEL`.
3. **Every model.** The result for every model (rows), one figure per experiment, dotted where it is not significant.
4. **How many models.** The multi-model median of the result (top row), then at each grid point how many models have a significant increase and how many a significant decrease; one figure per experiment.

This notebook:

| Section | Result | Test |
|---|---|---|
| 3. Quantile response | the change in Q05, Q50 and Q95 | none |
| 4. Change in width | the change in Q95 − Q05, and which tail did it | hist-nat bootstrap |

**By SAM phase.** Section 4 ends with the result in the seasons of each phase of the Southern Annular Mode alone (negative, neutral and positive, cut at hist-nat's terciles of the index: notebook calc_02, *SAM Phases*), the phases as rows.

**Reading the figures.** A quantity has the same colour scale in every figure (set once, under *Colour scales*). Every change is red for an increase and blue for a decrease, and dots mark where a change is not significant. The black lines on the maps of section 4 are the sea-ice edge (the 15% contour of the ensemble-mean concentration over the final 21 years, from notebook calc_04): hist-nat's dashed, the experiment's solid; the multi-model figures draw the edge of the models' median concentration. Without the sea ice they are left out. In the counts, the colour changes where more than half the models agree: reds turn brown (an increase) and blues purple (a decrease). The paper figures, which put the models together, are in notebook plot_05. The cells before each figure pick out its data and show them. Each figure is then drawn layer by layer with `extant/plots/antarctica.py`: `map_figure_size` and `map_axes` make the figure, its maps and colour bars, then each layer (a field, its dots, each ice edge) is drawn on those axes by its own call, a row of a different kind being a slice of them (`axes[2]`).

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame). Each saved result has the same name as in notebook calc_02. Until 14 October 2026 the four were one notebook, `04_forced_response_figures.ipynb` (in the git history); the version before that, from 7 October, is in `archive/forced_response_figures_2026-10-07.ipynb`.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path

import pandas as pd
import xarray as xr

from extant import config, response_change as rc, sam, sea_ice
from extant.config import ALPHA, FORCED_EXPERIMENTS, WINDOW
from extant import significance as sig
from extant.logger import setup_logging
from extant.plots import antarctica, charts, forced_response as fr, paper_panels, sea_ice as sip
from plotting_modules.constants import FORCING_COLORS, SEASONS

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)   # every model, experiment and season in one table

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

The settings every notebook shares are in `extant/config.py` (`ALPHA`, `FORCED_EXPERIMENTS`, `WINDOW`) and `plotting_modules/constants.py` (`SEASONS`). Only this notebook's own choices are here.

In [ ]:
VARIABLE = config.VARIABLES['tas']

#(t): The model and experiment of the "what goes in" and "one model" figures
EXAMPLE_MODEL = config.PLOT_SEL['model']
EXPERIMENT = config.PLOT_SEL['experiment']

#(t): The quantiles of section 3, and how they are labelled
RESPONSE_QUANTILES = [0.05, 0.50, 0.95]
QUANTILE_LABELS = {0.05: 'Q05', 0.50: 'Q50', 0.95: 'Q95'}

#(t): The SAM phases of 4.6, in the order of their rows, and how the rows are labelled
PHASES = list(sam.PHASES)
PHASE_LABELS = {phase: f'SAM\n{phase}' for phase in PHASES}

#(t): The share of the models that must have a significant change of one sign for a multi-model change to be robust
ROBUST = config.SIGNIFICANT_THRESHOLD


In [ ]:
#(t): Where notebooks calc_02 and calc_04 saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

### Colour scales

One scale per quantity, used by every figure that shows it (here and in notebooks plot_04 and plot_05), so the same colour always means the same value. Each is a `fr.Scale`: the contour levels, the colormap and the colour-bar label. Their ranges, per variable, are in `fr.RANGES`; notebook plot_05 (*Check the Colour Ranges*) says how much of each field falls beyond them.

In [ ]:
scales = fr.colour_scales(VARIABLE.name, alpha=ALPHA)

In [ ]:
#(t): The sea-ice zones (notebook calc_04, section 7), coloured as in notebook plot_02
zone_scale = fr.class_scale(sea_ice.ZONES, ['#f4f3ef', '#2c7fb8', '#e6550d', '#31a354', '#bdbdbd'], 'Zone')

In [ ]:
#(t): The legend entries of the maps: the dots, and the two ice edges (hist-nat's dashed, the experiment's solid)
not_significant_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none',
                                    label=f'Not significant (p ≥ {ALPHA:g})')
hist_nat_edge_handle = plt.Line2D([], [], color='k', linestyle='dashed', label="hist-nat's ice edge")
experiment_edge_handle = plt.Line2D([], [], color='k', label="The experiment's ice edge")

In [ ]:
#(t): What each one covers
pd.DataFrame({name: {'from': scale.levels[0], 'to': scale.levels[-1], 'colormap': getattr(scale.cmap, 'name', scale.cmap),
                     'label': scale.label}
              for name, scale in vars(scales).items()}).T

## Open the Saved Results

Run **one** of the next two sections. Both leave `RESULTS_DIR` and `ICE_RESULTS_DIR`, the folders notebook calc_02's and notebook calc_04's results are opened from.

### Sample data

In [ ]:
#(t): The sample's results: notebook calc_02's, and calc_04's sea ice
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'sample'
RESULTS_DIR, ICE_RESULTS_DIR

### Full data

In [ ]:
#(t): The full data's results: notebook calc_02's, and calc_04's sea ice
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
ICE_RESULTS_DIR = SCRATCH_DIR / 'results' / 'siconc' / 'full'
RESULTS_DIR, ICE_RESULTS_DIR

### Saved results

Everything sections 3 and 4 use, from `RESULTS_DIR`, one store per cell, loaded into memory.

In [ ]:
#(t): hist-nat's final-years quantiles
hist_nat_final_q_da = xr.open_zarr(RESULTS_DIR / 'hist_nat_final_quantiles.zarr')[VARIABLE.name].load()
hist_nat_final_q_da

In [ ]:
#(t): Every experiment's final-years quantiles
final_q_da = xr.open_zarr(RESULTS_DIR / 'final_quantiles.zarr')[VARIABLE.name].load()
final_q_da

In [ ]:
#(t): The change in Q05, Q50 and Q95
quantile_response_da = xr.open_zarr(RESULTS_DIR / 'quantile_response.zarr')['quantile_response'].load()
quantile_response_da

In [ ]:
#(t): The change in every 5th percentile, as an Antarctic mean
quantile_curve_da = xr.open_zarr(RESULTS_DIR / 'quantile_curve.zarr')['quantile_curve'].load()
quantile_curve_da

In [ ]:
#(t): The quantiles of the internal variability, in hist-nat and in the final years
width_quantiles_ds = xr.open_zarr(RESULTS_DIR / 'width_quantiles.zarr').load()
width_quantiles_ds

In [ ]:
#(t): The width test: the change in Q95 − Q05, and its p-value
qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'qrange_change.zarr').load()
qrange_change_ds

In [ ]:
#(t): The change in each tail, Q95 − Q50 and Q50 − Q05
tails_ds = xr.open_zarr(RESULTS_DIR / 'tails.zarr').load()
tails_ds

In [ ]:
#(t): The width test in each SAM phase
sam_qrange_change_ds = xr.open_zarr(RESULTS_DIR / 'sam_qrange_change.zarr').load()
sam_qrange_change_ds

In [ ]:
#(t): The tails in each SAM phase
sam_tails_ds = xr.open_zarr(RESULTS_DIR / 'sam_tails.zarr').load()
sam_tails_ds

In [ ]:
#(t): The number of models: the top of the colour bar of every count figure
N_MODELS = qrange_change_ds.sizes['model']
N_MODELS

### Sea ice

The sea-ice edge on the maps of section 4, and the sea-ice results of 4.7, come from notebook calc_04. Run **one** of the next two sections. Both leave `ice_climatology_da` (the ensemble-mean concentration over the final years), and `model_ice_da` and `median_ice_da` (`EXAMPLE_MODEL`'s concentration, and the models' median, for the edges of the one-model and multi-model maps); without sea ice they are `None` and the figures leave the edge out. *With sea ice* also opens the zones and the changes 4.7 uses; without it, 4.7 cannot run.

#### Without sea ice

Before notebook calc_04 has saved the sea ice (on the sample, that is when the notebooks are run in order).

In [ ]:
ice_climatology_da = model_ice_da = median_ice_da = None

#### With sea ice

In [ ]:
#(t): The ensemble-mean concentration over the final years (%), EXAMPLE_MODEL's, and the models' median (for the edges)
ice_climatology_da = xr.open_zarr(ICE_RESULTS_DIR / 'climatology.zarr')['siconc'].load()
model_ice_da = ice_climatology_da.sel(model=EXAMPLE_MODEL)
median_ice_da = ice_climatology_da.median('model')
median_ice_da

In [ ]:
#(t): Each cell's zone in the final years (the numbers of sea_ice.ZONES), every model, experiment and season
ice_zones_da = xr.open_zarr(ICE_RESULTS_DIR / 'zones.zarr')['zone'].load()
ice_zones_da

In [ ]:
#(t): The temperature changes averaged over each zone, and each zone's area (million km²)
tas_zone_means_ds = xr.open_zarr(ICE_RESULTS_DIR / 'zone_means.zarr').load()
tas_zone_means_ds

In [ ]:
#(t): The change in extent (million km²) and in the edge's latitude (degrees) against hist-nat
extent_change_ds = xr.open_zarr(ICE_RESULTS_DIR / 'extent_change.zarr').load()
extent_change_ds

In [ ]:
#(t): The temperature changes averaged over the cells with ice in hist-nat's final years
ice_region_change_ds = xr.open_zarr(ICE_RESULTS_DIR / 'ice_region_change.zarr').load()
ice_region_change_ds

## 3. Quantile Response: Q05, Q50 and Q95

How each part of the distribution has moved: the quantiles of each experiment's final 21 years (members and years pooled) minus hist-nat's over the same years (notebook calc_02, section 3):

$$
\Delta Q_p = Q_p\left(x^{\mathrm{exp}}_{\text{final } L \text{ years}}\right) - Q_p\left(x^{\mathrm{nat}}_{\text{final } L \text{ years}}\right).
$$

These are the values themselves, so the shift of the whole distribution is in every quantile. Equal changes in Q05, Q50 and Q95 are a pure shift; a bigger change in Q95 than in Q05 means the distribution has widened. There is no test of a single quantile: section 4 tests the width, Q95 − Q05.

### 3.1 What goes in

hist-nat's Q05, Q50 and Q95 and `EXPERIMENT`'s, both over the final 21 years, `EXAMPLE_MODEL`, and the change in each.

In [ ]:
#(t): hist-nat's quantiles over the final WINDOW years, EXAMPLE_MODEL, every season
hist_nat_q_da = hist_nat_final_q_da.sel(model=EXAMPLE_MODEL, season=SEASONS, quantile=RESPONSE_QUANTILES)

In [ ]:
#(t): EXPERIMENT's over the same years
experiment_q_da = final_q_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS,
                                 quantile=RESPONSE_QUANTILES)

In [ ]:
#(t): The change in each (quantile_response_da holds the same)
quantile_change_da = experiment_q_da - hist_nat_q_da
quantile_change_da

In [ ]:
#(t): hist-nat's Q05, Q50 and Q95 (rows) over the final WINDOW years
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=hist_nat_q_da.sizes['quantile'], n_cols=len(SEASONS),
                                                    n_colorbar_rows=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=hist_nat_q_da.sizes['quantile'], n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(hist_nat_q_da, 'quantile', 'season', axes=axes, cax=cax,
                                  **scales.temperature.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [QUANTILE_LABELS[q] for q in hist_nat_q_da['quantile'].values])
fig.suptitle(f'What goes into the quantile response: hist-nat (final {WINDOW} years), {EXAMPLE_MODEL}');

In [ ]:
#(t): EXPERIMENT's, over the same years
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=experiment_q_da.sizes['quantile'], n_cols=len(SEASONS),
                                                    n_colorbar_rows=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=experiment_q_da.sizes['quantile'], n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(experiment_q_da, 'quantile', 'season', axes=axes, cax=cax,
                                  **scales.temperature.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [QUANTILE_LABELS[q] for q in experiment_q_da['quantile'].values])
fig.suptitle(f'What goes into the quantile response: {EXPERIMENT} (final {WINDOW} years), {EXAMPLE_MODEL}');

In [ ]:
#(t): The change in each
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=quantile_change_da.sizes['quantile'], n_cols=len(SEASONS),
                                                    n_colorbar_rows=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=quantile_change_da.sizes['quantile'], n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(quantile_change_da, 'quantile', 'season', axes=axes, cax=cax,
                                  **scales.change.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [QUANTILE_LABELS[q] for q in quantile_change_da['quantile'].values])
fig.suptitle(f'The quantile response: {EXAMPLE_MODEL}, {EXPERIMENT} (final {WINDOW} years) against '
             f'hist-nat (the same years)');

### 3.2 One model

The change in Q05, Q50 and Q95 for every experiment of `EXAMPLE_MODEL`.

In [ ]:
#(t): EXAMPLE_MODEL's change in each quantile, every experiment and season
model_quantile_change_da = quantile_response_da.sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS, season=SEASONS)
model_quantile_change_da

In [ ]:
#(t): EXAMPLE_MODEL's change in each quantile, every experiment (rows), one figure per quantile
for quantile in RESPONSE_QUANTILES:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                        n_colorbar_rows=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(model_quantile_change_da.sel(quantile=quantile), 'experiment', 'season', axes=axes,
                                      cax=cax, **scales.change.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, FORCED_EXPERIMENTS)
    fig.suptitle(f'{EXAMPLE_MODEL}: change in {QUANTILE_LABELS[quantile]}, final {WINDOW} years against hist-nat')

### 3.3 Every model

The two ends of the distribution, Q05 and Q95 (the median moves with the mean, section 1); one figure per experiment.

In [ ]:
#(t): The change in Q05 for every model, experiment and season
all_q05_change_da = quantile_response_da.sel(quantile=0.05, experiment=FORCED_EXPERIMENTS, season=SEASONS)
all_q05_change_da

In [ ]:
#(t): Every model's change in Q05 (rows), one figure per experiment
models = all_q05_change_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_q05_change_da.sel(experiment=experiment), 'model', 'season', axes=axes,
                                      cax=cax, **scales.change.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.suptitle(f'Change in Q05, final {WINDOW} years against hist-nat, every model: {experiment}')

In [ ]:
#(t): and in Q95
all_q95_change_da = quantile_response_da.sel(quantile=0.95, experiment=FORCED_EXPERIMENTS, season=SEASONS)
all_q95_change_da

In [ ]:
#(t): Every model's change in Q95 (rows), one figure per experiment
models = all_q95_change_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_q95_change_da.sel(experiment=experiment), 'model', 'season', axes=axes,
                                      cax=cax, **scales.change.colours())

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.suptitle(f'Change in Q95, final {WINDOW} years against hist-nat, every model: {experiment}')

### 3.4 Every model, every quantile

With no test there is nothing to count, so instead the change in every 5th percentile, averaged over the area (south of 40°S), one line per model; experiments down, seasons across. A flat line is a pure shift; a rising line means the upper quantiles moved more than the lower ones, so the distribution widened.

In [ ]:
#(t): The area-mean change in every 5th percentile, every model, experiment and season
all_quantile_curve_da = quantile_curve_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): The same as percentiles (5 ... 95), for the x axis
all_percentile_curve_da = all_quantile_curve_da.assign_coords(
    percentile=('quantile', 100 * all_quantile_curve_da['quantile'].values)).swap_dims(quantile='percentile')
all_percentile_curve_da

In [ ]:
#(t): The area-mean change at every 5th percentile, one line per model (its shade of the experiment's colour, its line
#(t): style and marker): experiments down, seasons across, the y axis shared along each row
curve_models = list(all_percentile_curve_da['model'].values)
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1),
                 layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey='row', squeeze=False)
for ax in axes.ravel():
    ax.axhline(0, color='0.6', linewidth=0.8)
for i, experiment in enumerate(FORCED_EXPERIMENTS):
    model_styles = charts.model_lines(experiment, curve_models)
    for k, model in enumerate(curve_models):
        charts.lines_grid(all_percentile_curve_da.sel(experiment=experiment, model=model), 'percentile', None, 'season',
                          axes=axes[i], marker=charts.MODEL_MARKERS[k], markersize=3.5, **model_styles[model])

#(c): The axes are shared, so the ticks set on one panel are on all
axes[0, 0].set_xticks([0, 25, 50, 75, 100])
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
for ax, experiment in zip(axes[:, 0], FORCED_EXPERIMENTS):
    ax.set_ylabel(f'{experiment}\nChange ({VARIABLE.units})')
for ax in axes[-1]:
    ax.set_xlabel('Percentile')
#(c): One legend for every row: model k is the k-th shade of each row's colour, shown in historical's greys
legend_styles = charts.model_lines('historical', curve_models)
fig.legend(handles=[Line2D([], [], marker=charts.MODEL_MARKERS[k], markersize=3.5, **legend_styles[model])
                    for k, model in enumerate(curve_models)],
           loc='outside lower center', ncols=len(curve_models), frameon=False)
fig.suptitle(f'Change at every 5th percentile, area mean south of {abs(config.LAT_MAX)}°S, every model', x=0.01,
             ha='left');

## 4. Change in Width, and Which Tail

The width is $W = Q_{95} - Q_{05}$ of the **internal variability**: each member minus its experiment's forced response (the ensemble median smoothed with LOWESS), so that a trend within the 21 years does not widen it. The change is the width of each experiment's final 21 years minus hist-nat's over the same years (notebook calc_02, section 2):

$$
\Delta W = W\left(x'^{\,\mathrm{exp}}_{\text{final } L \text{ years}}\right) - W\left(x'^{\,\mathrm{nat}}_{\text{final } L \text{ years}}\right).
$$

It is tested against 1000 hist-nat bootstrap trials, each the width of 10 random hist-nat members over the same 21 years: significant where the experiment's width is outside the trials' central 95% ($p < 0.05$).

**Which tail.** The median splits the width into the lower tail $Q_{50} - Q_{05}$ and the upper tail $Q_{95} - Q_{50}$. Their changes add up to $\Delta W$, so the larger one says which tail did the widening or narrowing; for temperature, the lower tail is the cold one. Only the width is tested.

### 4.1 What goes in

The two tails and the width of hist-nat and of `EXPERIMENT`, both over the final 21 years, and the change in each; `EXAMPLE_MODEL`.

In [ ]:
#(t): hist-nat's Q05, Q50 and Q95 of the internal variability, over the final WINDOW years, EXAMPLE_MODEL, every season
hist_nat_width_q_da = width_quantiles_ds['hist_nat'].sel(model=EXAMPLE_MODEL, season=SEASONS)

In [ ]:
#(t): EXPERIMENT's, over the same years
experiment_width_q_da = width_quantiles_ds['final'].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)

In [ ]:
#(t): hist-nat's lower tail Q50 - Q05, upper tail Q95 - Q50 and width Q95 - Q05, on a new 'part' dim
hist_nat_parts_da = rc.width_parts(hist_nat_width_q_da)

In [ ]:
#(t): EXPERIMENT's
experiment_parts_da = rc.width_parts(experiment_width_q_da)

In [ ]:
#(t): The change in each (tails_ds and qrange_change_ds hold the same)
parts_change_da = experiment_parts_da - hist_nat_parts_da
parts_change_da

In [ ]:
#(t): How the rows are labelled
PART_LABELS = {part: f'{part.capitalize()}\n{formula}' for part, formula in rc.WIDTH_PARTS.items()}
PART_LABELS

In [ ]:
#(t): hist-nat's lower tail, upper tail and width (rows)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=hist_nat_parts_da.sizes['part'], n_cols=len(SEASONS),
                                                    n_colorbar_rows=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=hist_nat_parts_da.sizes['part'], n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(hist_nat_parts_da, 'part', 'season', axes=axes, cax=cax, **scales.spread.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [PART_LABELS[part] for part in hist_nat_parts_da['part'].values])
fig.suptitle(f'What goes into the change in width: hist-nat (final {WINDOW} years), {EXAMPLE_MODEL}, '
             f'internal variability');

In [ ]:
#(t): EXPERIMENT's
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=experiment_parts_da.sizes['part'], n_cols=len(SEASONS),
                                                    n_colorbar_rows=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=experiment_parts_da.sizes['part'], n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(experiment_parts_da, 'part', 'season', axes=axes, cax=cax, **scales.spread.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [PART_LABELS[part] for part in experiment_parts_da['part'].values])
fig.suptitle(f'What goes into the change in width: {EXPERIMENT} (final {WINDOW} years), {EXAMPLE_MODEL}, '
             f'internal variability');

In [ ]:
#(t): The change in each
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=parts_change_da.sizes['part'], n_cols=len(SEASONS),
                                                    n_colorbar_rows=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=parts_change_da.sizes['part'], n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(parts_change_da, 'part', 'season', axes=axes, cax=cax,
                                  **scales.spread_change.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, [PART_LABELS[part] for part in parts_change_da['part'].values])
fig.suptitle(f'The change in width: {EXAMPLE_MODEL}, {EXPERIMENT} (final {WINDOW} years) against hist-nat '
             f'(the same years), internal variability');

The test of that change: it is significant where it falls outside the central 95% of the hist-nat trials (the 2.5th to the 97.5th percentile of their changes).

In [ ]:
#(t): The width test for EXAMPLE_MODEL and EXPERIMENT: the change, the trials' range, the p-value
width_test_ds = qrange_change_ds.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)

In [ ]:
#(t): Not significant, where there is data
width_not_significant_da = ~width_test_ds['qrange_significant'] & width_test_ds['qrange_change'].notnull()
width_not_significant_da

In [ ]:
#(t): The width test: the change (dotted where not significant, with both ice edges), the hist-nat trials' 2.5th and
#(t): 97.5th percentiles (the same colour bar), and the p-value
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=4, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=4, n_cols=len(SEASONS))
#(c): The change's colour bar under rows 1-3 (the trials' range is on it too), the p-value's under row 4
width_cax = antarctica.colorbar_axis_below(axes[:3])
pvalue_cax = antarctica.colorbar_axis_below(axes[3])

#(c): Row 1: the change, its dots, and the ice edges (if the sea ice was opened)
antarctica.heatmaps_of_antarctica(width_test_ds['qrange_change'], None, 'season', axes=axes[0], cax=width_cax,
                                  **scales.spread_change.colours())
antarctica.not_significant_dots_grid(width_not_significant_da, None, 'season', axes=axes[0])
if model_ice_da is not None:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season', axes=axes[0],
                         linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT, season=SEASONS), None, 'season', axes=axes[0])
#(c): Rows 2 and 3: the trials' range, on the change's colour bar
antarctica.heatmaps_of_antarctica(width_test_ds['null_lower'], None, 'season', axes=axes[1],
                                  **scales.spread_change.colours())
antarctica.heatmaps_of_antarctica(width_test_ds['null_upper'], None, 'season', axes=axes[2],
                                  **scales.spread_change.colours())
#(c): Row 4: the p-value
antarctica.heatmaps_of_antarctica(width_test_ds['qrange_pvalue'], None, 'season', axes=axes[3], cax=pvalue_cax,
                                  **scales.pvalue.colours())

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, ['Change in width', 'hist-nat trials\n2.5th percentile', 'hist-nat trials\n97.5th percentile',
                             'p-value\nhist-nat bootstrap'])
fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center',
           ncols=3, fontsize=8)
fig.suptitle(f'The width test: {EXAMPLE_MODEL}, {EXPERIMENT}');

### 4.2 One model

For every experiment of `EXAMPLE_MODEL`: the change in each tail and the change in the width (their sum), dotted where the width test is not significant (4.1 shows the p-values).

In [ ]:
#(t): EXAMPLE_MODEL's tails, every experiment and season
model_tails_ds = tails_ds.sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS, season=SEASONS)
model_tails_ds

In [ ]:
#(t): Its width test
model_width_test_ds = qrange_change_ds.sel(model=EXAMPLE_MODEL, experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Not significant, where there is data
model_width_not_significant_da = ~model_width_test_ds['qrange_significant'] & model_width_test_ds['qrange_change'].notnull()
model_width_not_significant_da

In [ ]:
#(t): EXAMPLE_MODEL's change in the lower tail Q50 − Q05, every experiment (rows), with hist-nat's ice edge (dashed) and
#(t): each experiment's (solid)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_tails_ds['lower_tail_change'], 'experiment', 'season', axes=axes, cax=cax,
                                  **scales.spread_change.colours())
#(c): The ice edges, if the sea ice was opened: hist-nat's on every row (given the rows' experiments), each experiment's
if model_ice_da is not None:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS,
                                          drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                         'experiment', 'season', axes=axes, linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS), 'experiment', 'season',
                         axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'{EXAMPLE_MODEL}: change in the lower tail Q50 − Q05, final {WINDOW} years against hist-nat');

In [ ]:
#(t): EXAMPLE_MODEL's change in the upper tail Q95 − Q50, every experiment (rows), with hist-nat's ice edge (dashed) and
#(t): each experiment's (solid)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_tails_ds['upper_tail_change'], 'experiment', 'season', axes=axes, cax=cax,
                                  **scales.spread_change.colours())
#(c): The ice edges, if the sea ice was opened: hist-nat's on every row (given the rows' experiments), each experiment's
if model_ice_da is not None:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS,
                                          drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                         'experiment', 'season', axes=axes, linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS), 'experiment', 'season',
                         axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'{EXAMPLE_MODEL}: change in the upper tail Q95 − Q50, final {WINDOW} years against hist-nat');

In [ ]:
#(t): EXAMPLE_MODEL's change in the width Q95 − Q05, every experiment (rows), with hist-nat's ice edge (dashed) and each
#(t): experiment's (solid)
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS),
                                                    n_colorbar_rows=1,
                                                    n_legends=1), layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(model_width_test_ds['qrange_change'], 'experiment', 'season', axes=axes, cax=cax,
                                  **scales.spread_change.colours())
antarctica.not_significant_dots_grid(model_width_not_significant_da, 'experiment', 'season', axes=axes)
#(c): The ice edges, if the sea ice was opened: hist-nat's on every row (given the rows' experiments), each experiment's
if model_ice_da is not None:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS,
                                          drop=True).expand_dims(experiment=FORCED_EXPERIMENTS),
                         'experiment', 'season', axes=axes, linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=FORCED_EXPERIMENTS, season=SEASONS), 'experiment', 'season',
                         axes=axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, FORCED_EXPERIMENTS)
fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center',
           ncols=3, fontsize=8)
fig.suptitle(f'{EXAMPLE_MODEL}: change in the width Q95 − Q05, final {WINDOW} years against hist-nat');

### 4.3 Every model

One figure per experiment: every model down, every season across.

In [ ]:
#(t): The change in width for every model, experiment and season
all_width_change_da = qrange_change_ds['qrange_change'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Significant (p < ALPHA in notebook calc_02)
all_width_significant_da = qrange_change_ds['qrange_significant'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Not significant, where there is data
all_width_not_significant_da = ~all_width_significant_da & all_width_change_da.notnull()
all_width_not_significant_da

In [ ]:
#(t): Every model (rows), one figure per experiment, dotted where the width change is not significant, with each
#(t): model's ice edges
models = all_width_change_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_width_change_da.sel(experiment=experiment), 'model', 'season', axes=axes,
                                      cax=cax, **scales.spread_change.colours())
    antarctica.not_significant_dots_grid(all_width_not_significant_da.sel(experiment=experiment), 'model', 'season',
                                         axes=axes)
    #(c): The ice edges in the rows' order (empty for a model without sea ice), if the sea ice was opened
    if ice_climatology_da is not None:
        rows_ice_da = ice_climatology_da.reindex(model=models).sel(season=SEASONS)
        antarctica.ice_edges(rows_ice_da.sel(experiment='hist-nat'), 'model', 'season', axes=axes, linestyle='dashed')
        antarctica.ice_edges(rows_ice_da.sel(experiment=experiment), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle],
               loc='outside lower center', ncols=3,
               fontsize=8)
    fig.suptitle(f'Change in width Q95 − Q05, final {WINDOW} years against hist-nat, every model: {experiment}')

### 4.4 How many models

The multi-model median of the change in width (top row), then the number of models with a significantly wider distribution and a significantly narrower one. One figure per experiment.

In [ ]:
#(t): The multi-model median of the change in width
median_width_change_da = all_width_change_da.median('model')
median_width_change_da

In [ ]:
#(t): How many models are significantly wider, and how many narrower
width_counts_da = sig.significant_counts(all_width_change_da, all_width_significant_da)

In [ ]:
#(t): The decreases (narrower) as negative counts, so that one colour bar (blues for narrower, reds for wider) serves
#(t): both rows of counts
signed_width_counts_da = xr.concat([width_counts_da.sel(direction=['increase']),
                                    -width_counts_da.sel(direction=['decrease'])], dim='direction')
signed_width_counts_da

In [ ]:
#(t): Their colours: blues and reds up to half the models, purples and browns beyond, white for none
width_count_colours = fr.count_scale(N_MODELS, f'Number of models significantly narrower or wider, '
                                               f'p < {ALPHA:g}').colours()

In [ ]:
#(t): The multi-model median (top row), then how many models are significantly wider and narrower, one figure per
#(t): experiment, with the ice edges of the models' median concentration
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=3, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1),
                     layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=3, n_cols=len(SEASONS))
    #(c): The median's colour bar under row 1, the counts' under rows 2 and 3
    median_cax = antarctica.colorbar_axis_below(axes[0])
    count_cax = antarctica.colorbar_axis_below(axes[1:])

    antarctica.heatmaps_of_antarctica(median_width_change_da.sel(experiment=experiment), None, 'season', axes=axes[0],
                                      cax=median_cax, **scales.spread_change.colours())
    antarctica.heatmaps_of_antarctica(signed_width_counts_da.sel(experiment=experiment), 'direction', 'season',
                                      axes=axes[1:], cax=count_cax, **width_count_colours)
    #(c): Both ice edges on every row, if the sea ice was opened
    if median_ice_da is not None:
        for row_axes in axes:
            antarctica.ice_edges(median_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season',
                                 axes=row_axes, linestyle='dashed')
            antarctica.ice_edges(median_ice_da.sel(experiment=experiment, season=SEASONS), None, 'season',
                                 axes=row_axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, ['Multi-model\nmedian', 'Significantly\nwider', 'Significantly\nnarrower'])
    fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
    fig.suptitle(f'How many models have a significant change in width: {experiment}')

### 4.5 Which tail, every model

The upper tail's change minus the lower tail's: red where the upper (warm) tail stretched more or shrank less, blue where the lower (cold) tail did. Dotted where the width change itself is not significant; one figure per experiment.

In [ ]:
#(t): The tail asymmetry for every model, experiment and season
all_tail_asymmetry_da = tails_ds['tail_asymmetry'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)
all_tail_asymmetry_da

In [ ]:
#(t): Every model (rows), one figure per experiment, dotted where the width change is not significant, with each
#(t): model's ice edges
models = all_tail_asymmetry_da['model'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(all_tail_asymmetry_da.sel(experiment=experiment), 'model', 'season', axes=axes,
                                      cax=cax, **scales.tail.colours())
    antarctica.not_significant_dots_grid(all_width_not_significant_da.sel(experiment=experiment), 'model', 'season',
                                         axes=axes)
    #(c): The ice edges in the rows' order (empty for a model without sea ice), if the sea ice was opened
    if ice_climatology_da is not None:
        rows_ice_da = ice_climatology_da.reindex(model=models).sel(season=SEASONS)
        antarctica.ice_edges(rows_ice_da.sel(experiment='hist-nat'), 'model', 'season', axes=axes, linestyle='dashed')
        antarctica.ice_edges(rows_ice_da.sel(experiment=experiment), 'model', 'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, models)
    fig.legend(handles=[not_significant_handle, hist_nat_edge_handle, experiment_edge_handle],
               loc='outside lower center', ncols=3,
               fontsize=8)
    fig.suptitle(f'Which tail changed more: upper-tail minus lower-tail change, every model: {experiment}')

### 4.6 By SAM phase

The change in width in the seasons of each SAM phase alone (notebook calc_02, section 2, *By SAM phase*), and which tail did it (notebook calc_02, section 7): for `EXAMPLE_MODEL`, dotted where it is not significant; the multi-model median, dotted where it is not robust; how many models; and the multi-model median of which tail.

In [ ]:
#(t): EXAMPLE_MODEL's change in width in each phase, every experiment and season
model_phase_width_change_da = sam_qrange_change_ds['qrange_change'].sel(model=EXAMPLE_MODEL,
                                                                        experiment=FORCED_EXPERIMENTS,
                                                                         season=SEASONS)

In [ ]:
#(t): Not significant, where there is data
model_phase_width_not_significant_da = (~sam_qrange_change_ds['qrange_significant'].sel(model=EXAMPLE_MODEL,
                                                                                        experiment=FORCED_EXPERIMENTS,
                                                                                         season=SEASONS)
                                        & model_phase_width_change_da.notnull())
model_phase_width_not_significant_da

In [ ]:
#(t): EXAMPLE_MODEL's change in width in each SAM phase (rows), one figure per experiment, dotted where not significant
phases = model_phase_width_change_da['phase'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(phases), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(phases), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(model_phase_width_change_da.sel(experiment=experiment), 'phase', 'season',
                                      axes=axes, cax=cax, **scales.spread_change.colours())
    antarctica.not_significant_dots_grid(model_phase_width_not_significant_da.sel(experiment=experiment), 'phase',
                                         'season', axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in phases])
    fig.legend(handles=[not_significant_handle], loc='outside lower center', fontsize=8)
    fig.suptitle(f'{EXAMPLE_MODEL}: change in width Q95 − Q05 in each SAM phase, final {WINDOW} years against '
                 f'hist-nat: {experiment}')

In [ ]:
#(t): The change in width in each phase, every model, experiment and season
all_phase_width_change_da = sam_qrange_change_ds['qrange_change'].sel(experiment=FORCED_EXPERIMENTS, season=SEASONS)

In [ ]:
#(t): Significant (p < ALPHA in notebook calc_02)
all_phase_width_significant_da = sam_qrange_change_ds['qrange_significant'].sel(experiment=FORCED_EXPERIMENTS,
                                                                                season=SEASONS)

In [ ]:
#(t): The multi-model median
median_phase_width_change_da = all_phase_width_change_da.median('model')
median_phase_width_change_da

In [ ]:
#(t): Where the models agree: +1 (-1) where at least ROBUST of them are significantly wider (narrower)
phase_width_robust_da = sig.robust_sign(all_phase_width_change_da, all_phase_width_significant_da, threshold=ROBUST)

In [ ]:
#(t): Not robust, where there is data
phase_width_not_robust_da = phase_width_robust_da == 0
phase_width_not_robust_da

In [ ]:
#(t): The multi-model median in each SAM phase (rows), one figure per experiment, dotted where the width change is not
#(t): robust
phases = median_phase_width_change_da['phase'].values
not_robust_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none', label='Not robust (fewer than two-thirds\nof the models significant with one sign)')
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(phases), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(phases), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(median_phase_width_change_da.sel(experiment=experiment), 'phase', 'season', axes=axes,
                                      cax=cax, **scales.spread_change.colours())
    antarctica.not_significant_dots_grid(phase_width_not_robust_da.sel(experiment=experiment), 'phase', 'season',
                                         axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in phases])
    fig.legend(handles=[not_robust_handle], loc='outside lower center', fontsize=8)
    fig.suptitle(f'Change in width Q95 − Q05 in each SAM phase, multi-model median: {experiment}')

In [ ]:
#(t): How many models are significantly wider, and how many narrower, in each phase
phase_width_counts_da = sig.significant_counts(all_phase_width_change_da, all_phase_width_significant_da)

In [ ]:
#(t): The decreases (narrower) as negative counts, as in 4.4
signed_phase_width_counts_da = xr.concat([phase_width_counts_da.sel(direction=['increase']),
                                          -phase_width_counts_da.sel(direction=['decrease'])], dim='direction')
signed_phase_width_counts_da

In [ ]:
#(t): How many models are significantly wider and narrower in each SAM phase: two rows per phase, one colour bar
phases = signed_phase_width_counts_da['phase'].values
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=2 * len(phases), n_cols=len(SEASONS),
                                                        n_colorbar_rows=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=2 * len(phases), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    #(c): Each phase's two rows, wider then narrower; the colour bar drawn once, with the first phase
    for k, phase in enumerate(phases):
        antarctica.heatmaps_of_antarctica(signed_phase_width_counts_da.sel(experiment=experiment, phase=phase),
                                          'direction', 'season', axes=axes[2 * k:2 * k + 2],
                                          cax=cax if k == 0 else None, **width_count_colours)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [f'SAM {phase}\n{direction}' for phase in phases for direction in ['wider', 'narrower']])
    fig.suptitle(f'How many models have a significant change in width in each SAM phase: {experiment}')

In [ ]:
#(t): Which tail: the multi-model median of the upper tail's change minus the lower tail's, in each phase
median_phase_tail_asymmetry_da = sam_tails_ds['tail_asymmetry'].sel(experiment=FORCED_EXPERIMENTS,
                                                                    season=SEASONS).median('model')
median_phase_tail_asymmetry_da

In [ ]:
#(t): The multi-model median in each SAM phase (rows), one figure per experiment, dotted where the width change is not
#(t): robust
phases = median_phase_tail_asymmetry_da['phase'].values
not_robust_handle = plt.Line2D([], [], color='k', marker='.', linestyle='none', label='Width change not robust')
for experiment in FORCED_EXPERIMENTS:
    fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(phases), n_cols=len(SEASONS), n_colorbar_rows=1,
                                                        n_legends=1), layout='constrained')
    axes = antarctica.map_axes(fig, n_rows=len(phases), n_cols=len(SEASONS))
    cax = antarctica.colorbar_axis_below(axes)

    antarctica.heatmaps_of_antarctica(median_phase_tail_asymmetry_da.sel(experiment=experiment), 'phase', 'season', axes=axes,
                                      cax=cax, **scales.tail.colours())
    antarctica.not_significant_dots_grid(phase_width_not_robust_da.sel(experiment=experiment), 'phase', 'season',
                                         axes=axes)

    antarctica.label_columns(axes, SEASONS)
    antarctica.label_rows(axes, [PHASE_LABELS[phase] for phase in phases])
    fig.legend(handles=[not_robust_handle], loc='outside lower center', fontsize=8)
    fig.suptitle(f'Which tail changed more in each SAM phase: upper-tail minus lower-tail change, multi-model median: '
                 f'{experiment}')

### 4.7 Sea ice and the change in width

Where the ice changed, and how the width and each tail changed there (notebook calc_04, section 7), with the zones of 1.6 (notebook plot_03a): open water in both, ice in both, ice lost, ice gained, or the continent. The expectation (Bracegirdle et al., 2024): where the ice is lost, the cold tail warms more than the warm one, so the lower tail shrinks and the distribution narrows. First what goes in for `EXAMPLE_MODEL` and `EXPERIMENT`; then the change in width and in each tail averaged over each zone, every model; then, one point per model and forcing, the change in extent against the change in width over hist-nat's ice. Needs the sea ice (*With sea ice* above).

In [ ]:
#(t): EXAMPLE_MODEL's zones for EXPERIMENT, every season
model_zones_da = ice_zones_da.sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT, season=SEASONS)
model_zones_da

In [ ]:
#(t): EXAMPLE_MODEL's change in each tail and in the width for EXPERIMENT
model_lower_tail_change_da = tails_ds['lower_tail_change'].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                               season=SEASONS)
model_upper_tail_change_da = tails_ds['upper_tail_change'].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                               season=SEASONS)
model_width_change_da = qrange_change_ds['qrange_change'].sel(model=EXAMPLE_MODEL, experiment=EXPERIMENT,
                                                              season=SEASONS)
model_width_change_da

In [ ]:
#(t): What goes in: the zones, and the change in each tail and in the width (one colour bar), each with both ice edges
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=4, n_cols=len(SEASONS), n_colorbar_rows=2, n_legends=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=4, n_cols=len(SEASONS))
#(c): The zones' colour bar under row 1, the changes' under rows 2-4
zone_cax = antarctica.colorbar_axis_below(axes[0])
change_cax = antarctica.colorbar_axis_below(axes[1:])

antarctica.heatmaps_of_antarctica(model_zones_da, None, 'season', axes=axes[0], cax=zone_cax, **zone_scale.colours())
antarctica.heatmaps_of_antarctica(model_lower_tail_change_da, None, 'season', axes=axes[1], cax=change_cax,
                                  **scales.spread_change.colours())
antarctica.heatmaps_of_antarctica(model_upper_tail_change_da, None, 'season', axes=axes[2],
                                  **scales.spread_change.colours())
antarctica.heatmaps_of_antarctica(model_width_change_da, None, 'season', axes=axes[3], **scales.spread_change.colours())
#(c): Both ice edges on every row
for row_axes in axes:
    antarctica.ice_edges(model_ice_da.sel(experiment='hist-nat', season=SEASONS), None, 'season', axes=row_axes,
                         linestyle='dashed')
    antarctica.ice_edges(model_ice_da.sel(experiment=EXPERIMENT, season=SEASONS), None, 'season', axes=row_axes)

antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, ['Zone', f'Δ lower tail\n{rc.WIDTH_PARTS["lower tail"]}',
                             f'Δ upper tail\n{rc.WIDTH_PARTS["upper tail"]}', f'Δ width\n{rc.WIDTH_PARTS["width"]}'])
fig.legend(handles=[hist_nat_edge_handle, experiment_edge_handle], loc='outside lower center', ncols=2, fontsize=8)
fig.suptitle(f'Where the ice changed, and the width: {EXAMPLE_MODEL}, {EXPERIMENT}, final {WINDOW} years');

In [ ]:
#(t): The change in width and in each tail averaged over each zone, every model, experiment and season (°C)
tas_zone_means_ds[['width', 'lower tail', 'upper tail']]

In [ ]:
#(t): The legend of the zone figures: each model's marker, and the bar of their median
zone_legend_handles = (charts.model_legend_handles(tas_zone_means_ds['model'].values)
                       + [Patch(color='0.6', alpha=0.5, label='Multi-model median')])

In [ ]:
#(t): The change in width in each zone: experiments down, seasons across; a bar for the median of the models and a
#(t): marker for each model, in the experiment's colour
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(tas_zone_means_ds['width'].sel(experiment=experiment, season=SEASONS), 'zone', None,
                           'season', axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ(Q95 − Q05) (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle(f'Change in the width in each zone, final {WINDOW} years against hist-nat', x=0.01, ha='left');

In [ ]:
#(t): ... the change in the lower (cold) tail, Q50 − Q05: negative where the cold extremes warm more than the median
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(tas_zone_means_ds['lower tail'].sel(experiment=experiment, season=SEASONS), 'zone', None, 'season',
                          axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ(Q50 − Q05) (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle('Change in the lower (cold) tail in each zone: negative where the cold extremes warm more than the median',
             x=0.01, ha='left');

In [ ]:
#(t): ... the change in the upper (warm) tail, Q95 − Q50
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=len(FORCED_EXPERIMENTS), n_cols=len(SEASONS), n_legends=1,
                                                  panel_height=1.9), layout='constrained')
axes = fig.add_gridspec(len(FORCED_EXPERIMENTS), len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
for row_axes, experiment in zip(axes, FORCED_EXPERIMENTS):
    charts.model_dots_grid(tas_zone_means_ds['upper tail'].sel(experiment=experiment, season=SEASONS), 'zone', None, 'season',
                          axes=row_axes, color=FORCING_COLORS[experiment])
    row_axes[0].set_ylabel(f'{experiment}\nΔ(Q95 − Q50) (°C)')
for ax, season in zip(axes[0], SEASONS):
    ax.set_title(season, loc='left')
fig.legend(handles=zone_legend_handles, loc='outside lower center', ncols=len(zone_legend_handles), frameon=False)
fig.suptitle('Change in the upper (warm) tail in each zone', x=0.01, ha='left');

In [ ]:
#(t): The change in extent (million km²) and the change in width over hist-nat's ice (°C), every model and experiment
extent_change_da = extent_change_ds['extent_change']
ice_region_width_change_da = ice_region_change_ds['width']
ice_region_width_change_da

In [ ]:
#(t): The change in extent against the change in width over hist-nat's ice: one point per model (marker) and forcing
#(t): (colour), one panel per season; r is the correlation over all the points
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=len(SEASONS), n_legends=2), layout='constrained')
#(c): The panels, then a row for each legend (the layout gives each the room it needs)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 0.01, 0.01])
axes = grid[0].subgridspec(1, len(SEASONS)).subplots(sharex=True, sharey=True, squeeze=False)
legend_axes = [fig.add_subplot(grid[row]) for row in (1, 2)]
for legend_ax in legend_axes:
    legend_ax.axis('off')
for ax, season in zip(axes[0], SEASONS):
    charts.plot_scatter_by_forcing(extent_change_da.sel(experiment=FORCED_EXPERIMENTS, season=season),
                                         ice_region_width_change_da.sel(experiment=FORCED_EXPERIMENTS,
                                                                        season=season), ax=ax)
    ax.set_title(season, loc='left')
    ax.set_xlabel('Δ extent (million km²)')
axes[0, 0].set_ylabel("Δ(Q95 − Q05) over hist-nat's ice (°C)")
legend_axes[0].legend(handles=charts.forcing_legend_handles(FORCED_EXPERIMENTS), loc='center',
                      ncols=len(FORCED_EXPERIMENTS), frameon=False)
legend_axes[1].legend(handles=charts.model_legend_handles(extent_change_da['model'].values),
                      loc='center', ncols=extent_change_da.sizes['model'], frameon=False)
fig.suptitle(f"Change in extent against the change in width over hist-nat's ice, final {WINDOW} years", x=0.01,
             ha='left');